# CNN-Based Waste Image Classification for Recycling Automation
**MSc Software Engineering – Pattern Recognition | Phase 2**  
**Student: Jasmine Praween**  
**Dataset:** Garbage Classification – [Kaggle](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification)  

---
## Notebook Contents
1. Install & Import Libraries
2. Dataset Loading & Exploration
3. Preprocessing & Data Augmentation
4. Custom CNN Model – Build, Train, Evaluate
5. Transfer Learning – MobileNetV2 Fine-Tuning
6. Model Comparison
7. Bonus: Grad-CAM Visualisation

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries (run once on Kaggle)
# !pip install -q tensorflow matplotlib seaborn scikit-learn

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# Scikit-learn
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## Step 2: Dataset Loading & Exploration

In [ ]:
# ============================================================
# CONFIGURE DATASET PATH
# ============================================================
DATASET_PATH = '/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification'
print("Dataset path:", DATASET_PATH)
print("Exists:", os.path.exists(DATASET_PATH))
print("Contents:", os.listdir(DATASET_PATH))


In [ ]:
import os
print(os.listdir('/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification'))


In [ ]:
# ============================================================
# Count images per class
# ============================================================
class_counts = {}
for class_name in CLASS_NAMES:
    class_dir = os.path.join(DATASET_PATH, class_name)
    if os.path.exists(class_dir):
        count = len([f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        class_counts[class_name] = count

print('\nClass Distribution:')
for c, n in class_counts.items():
    print(f'  {c:<12}: {n} images')
print(f'\n  TOTAL       : {sum(class_counts.values())} images')

# Plot class distribution
plt.figure(figsize=(9, 4))
colors = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800','#607D8B']
bars = plt.bar(class_counts.keys(), class_counts.values(), color=colors, edgecolor='white', linewidth=0.8)
for bar, count in zip(bars, class_counts.values()):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5, str(count),
             ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.title('Class Distribution in Garbage Classification Dataset', fontsize=13, fontweight='bold')
plt.xlabel('Waste Category', fontsize=11)
plt.ylabel('Number of Images', fontsize=11)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: class_distribution.png')

In [ ]:
# ============================================================
# Display sample images from each class
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATASET_PATH, class_name)
    if os.path.exists(class_dir):
        img_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        if img_files:
            img_path = os.path.join(class_dir, img_files[0])
            img = plt.imread(img_path)
            axes[i].imshow(img)
            axes[i].set_title(f'{class_name.capitalize()}\n({class_counts.get(class_name, 0)} images)',
                              fontsize=11, fontweight='bold')
            axes[i].axis('off')

plt.suptitle('Sample Images from Each Waste Category', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

## Step 3: Preprocessing & Data Augmentation

In [ ]:
# ============================================================
# Custom CNN Data Generators
# Normalise: pixel values divided by 255 → [0, 1]
# Augmentation applied ONLY to training data
# ============================================================

# Training generator WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,          # 80% train, 20% validation
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation/test generator WITHOUT augmentation
val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Training set
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

# Validation set
val_generator = val_test_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

print(f'Training samples  : {train_generator.samples}')
print(f'Validation samples: {val_generator.samples}')
print(f'Class indices     : {train_generator.class_indices}')

In [ ]:
# ============================================================
# Visualise augmented samples
# ============================================================
aug_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Get one sample image (paper class)
sample_class = 'paper'
sample_dir = os.path.join(DATASET_PATH, sample_class)
sample_img_file = [f for f in os.listdir(sample_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))][0]
sample_img = tf.keras.preprocessing.image.load_img(
    os.path.join(sample_dir, sample_img_file), target_size=IMG_SIZE)
sample_arr = tf.keras.preprocessing.image.img_to_array(sample_img)
sample_arr = sample_arr.reshape((1,) + sample_arr.shape)

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
axes[0].imshow(sample_arr[0].astype('uint8'))
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for i, batch in enumerate(aug_gen.flow(sample_arr, batch_size=1, seed=i)):
    axes[i+1].imshow(batch[0])
    axes[i+1].set_title(f'Augmented {i+1}', fontsize=9)
    axes[i+1].axis('off')
    if i >= 4:
        break

plt.suptitle('Data Augmentation Examples (Paper Class)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('augmentation_examples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: augmentation_examples.png')

## Step 4: Custom CNN Model

In [ ]:
# ============================================================
# Build Custom CNN Architecture
# 3 Convolutional blocks + 2 Dense layers
# Batch Normalisation + Dropout for regularisation
# ============================================================

def build_custom_cnn(input_shape=(224, 224, 3), num_classes=6):
    """
    Custom CNN for waste image classification.
    Architecture:
      - 3 x Conv blocks: Conv2D → BatchNorm → MaxPool
      - Flatten → Dense(256) → Dropout(0.5)
      - Dense(128) → Dropout(0.3)
      - Output: Dense(num_classes, Softmax)
    """
    model = models.Sequential(name='Custom_CNN')

    # --- Input
    model.add(layers.InputLayer(input_shape=input_shape))

    # --- Convolutional Block 1 ---
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3, 3), padding='same', activation='relu', name='conv1_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), name='pool1'))

    # --- Convolutional Block 2 ---
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3, 3), padding='same', activation='relu', name='conv2_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), name='pool2'))

    # --- Convolutional Block 3 ---
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3_1'))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3, 3), padding='same', activation='relu', name='conv3_2'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((2, 2), name='pool3'))

    # --- Dense Layers ---
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation='relu', name='dense1'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation='relu', name='dense2'))
    model.add(layers.Dropout(0.3))

    # --- Output Layer ---
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))

    return model

# Build model
cnn_model = build_custom_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)

# Compile
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# ============================================================
# Callbacks: Early stopping + Learning rate scheduler
# ============================================================
cnn_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_custom_cnn.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ============================================================
# TRAIN Custom CNN
# ============================================================
print('Training Custom CNN...')
cnn_history = cnn_model.fit(
    train_generator,
    epochs=EPOCHS_CNN,
    validation_data=val_generator,
    callbacks=cnn_callbacks,
    verbose=1
)
print('Custom CNN training complete.')

In [ ]:
# ============================================================
# Plot Training Curves for Custom CNN
# ============================================================
def plot_training_curves(history, model_name, save_prefix):
    """Plot and save accuracy and loss training curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    epochs_ran = range(1, len(history.history['accuracy']) + 1)

    # Accuracy
    ax1.plot(epochs_ran, history.history['accuracy'],     'b-o', markersize=4, label='Train Accuracy')
    ax1.plot(epochs_ran, history.history['val_accuracy'], 'r-o', markersize=4, label='Val Accuracy')
    ax1.set_title(f'{model_name} – Accuracy', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(epochs_ran, history.history['loss'],     'b-o', markersize=4, label='Train Loss')
    ax2.plot(epochs_ran, history.history['val_loss'], 'r-o', markersize=4, label='Val Loss')
    ax2.set_title(f'{model_name} – Loss', fontsize=12, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{save_prefix}_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_prefix}_training_curves.png')

plot_training_curves(cnn_history, 'Custom CNN', 'custom_cnn')

In [ ]:
# ============================================================
# Evaluate Custom CNN on Validation Set
# ============================================================
def evaluate_model(model, generator, class_names, model_name, save_prefix):
    """
    Full evaluation:
    - Accuracy & loss on generator
    - Confusion matrix heatmap
    - Classification report
    Returns: accuracy score
    """
    generator.reset()

    # Predictions
    y_pred_prob = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = generator.classes

    # Accuracy
    val_loss, val_acc = model.evaluate(generator, verbose=0)
    print(f'\n{model_name} – Evaluation Results')
    print(f'  Validation Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)')
    print(f'  Validation Loss     : {val_loss:.4f}')

    # Classification report
    print(f'\nClassification Report ({model_name}):')
    print(classification_report(y_true, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, linecolor='gray')
    plt.title(f'{model_name} – Confusion Matrix', fontsize=13, fontweight='bold')
    plt.ylabel('True Label', fontsize=11)
    plt.xlabel('Predicted Label', fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_prefix}_confusion_matrix.png')

    return val_acc, y_pred_prob, y_true

cnn_acc, cnn_probs, cnn_true = evaluate_model(
    cnn_model, val_generator, CLASS_NAMES, 'Custom CNN', 'custom_cnn'
)

## Step 5: Transfer Learning – MobileNetV2

In [ ]:
# ============================================================
# MobileNetV2 Data Generators
# Uses MobileNetV2-specific preprocessing ([-1, 1] range)
# ============================================================
tl_train_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

tl_val_datagen = ImageDataGenerator(
    preprocessing_function=mobilenet_preprocess,
    validation_split=0.2
)

tl_train_gen = tl_train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

tl_val_gen = tl_val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)

print(f'TL Training samples  : {tl_train_gen.samples}')
print(f'TL Validation samples: {tl_val_gen.samples}')

In [ ]:
# ============================================================
# Build MobileNetV2 Transfer Learning Model
# Phase 1: Freeze base → train top layers only
# ============================================================
def build_mobilenetv2_model(input_shape=(224, 224, 3), num_classes=6):
    """
    MobileNetV2 transfer learning model.
    Base: MobileNetV2 pretrained on ImageNet (frozen)
    Top: GlobalAvgPool → Dense(128, ReLU) → Dropout(0.3) → Dense(num_classes, Softmax)
    """
    # Load pre-trained base (excluding top classification head)
    base_model = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    base_model.trainable = False   # Freeze base layers

    # Build top layers
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x = layers.Dense(128, activation='relu', name='dense1')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='MobileNetV2_TL')
    return model, base_model

tl_model, base_model = build_mobilenetv2_model(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)

# Compile – Phase 1
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

tl_model.summary()

In [ ]:
# ============================================================
# Phase 1: Train top layers (base frozen) – 15 epochs
# ============================================================
tl_callbacks_phase1 = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

print('Phase 1: Training top layers (base frozen)...')
tl_history_p1 = tl_model.fit(
    tl_train_gen,
    epochs=15,
    validation_data=tl_val_gen,
    callbacks=tl_callbacks_phase1,
    verbose=1
)
print('Phase 1 complete.')

In [ ]:
# ============================================================
# Phase 2: Fine-tune top 30 layers of base model
# ============================================================
print(f'Total layers in base model: {len(base_model.layers)}')

# Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower learning rate
tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

tl_callbacks_phase2 = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
    callbacks.ModelCheckpoint('best_mobilenetv2.keras', monitor='val_accuracy', save_best_only=True, verbose=1),
]

print('Phase 2: Fine-tuning top layers...')
tl_history_p2 = tl_model.fit(
    tl_train_gen,
    epochs=EPOCHS_TL,
    validation_data=tl_val_gen,
    callbacks=tl_callbacks_phase2,
    verbose=1
)
print('Phase 2 fine-tuning complete.')

In [ ]:
# Plot MobileNetV2 training curves (Phase 2 fine-tuning)
plot_training_curves(tl_history_p2, 'MobileNetV2 (Fine-tuning)', 'mobilenetv2')

In [ ]:
# Evaluate MobileNetV2
tl_acc, tl_probs, tl_true = evaluate_model(
    tl_model, tl_val_gen, CLASS_NAMES, 'MobileNetV2', 'mobilenetv2'
)

## Step 6: Model Comparison

In [ ]:
# ============================================================
# Side-by-side comparison: Custom CNN vs MobileNetV2
# ============================================================
from sklearn.metrics import precision_score, recall_score, f1_score

def get_metrics(y_true, y_pred_prob):
    y_pred = np.argmax(y_pred_prob, axis=1)
    acc  = np.mean(y_pred == y_true)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return acc, prec, rec, f1

cnn_metrics = get_metrics(cnn_true, cnn_probs)
tl_metrics  = get_metrics(tl_true,  tl_probs)

# Comparison table
comparison_df = pd.DataFrame({
    'Model'    : ['Custom CNN', 'MobileNetV2 (Transfer Learning)'],
    'Accuracy' : [f'{cnn_metrics[0]*100:.2f}%', f'{tl_metrics[0]*100:.2f}%'],
    'Precision': [f'{cnn_metrics[1]*100:.2f}%', f'{tl_metrics[1]*100:.2f}%'],
    'Recall'   : [f'{cnn_metrics[2]*100:.2f}%', f'{tl_metrics[2]*100:.2f}%'],
    'F1-Score' : [f'{cnn_metrics[3]*100:.2f}%', f'{tl_metrics[3]*100:.2f}%'],
})

print('\n===== MODEL COMPARISON =====')
print(comparison_df.to_string(index=False))

# Bar chart comparison
metrics_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
cnn_vals = [m*100 for m in cnn_metrics]
tl_vals  = [m*100 for m in tl_metrics]

x = np.arange(len(metrics_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, cnn_vals, width, label='Custom CNN',    color='#2196F3', alpha=0.85)
bars2 = ax.bar(x + width/2, tl_vals,  width, label='MobileNetV2',   color='#4CAF50', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, color='#1565C0')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9, color='#2E7D32')

ax.set_xlabel('Metric', fontsize=11)
ax.set_ylabel('Score (%)', fontsize=11)
ax.set_title('Custom CNN vs MobileNetV2 – Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_labels)
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_comparison.png')

In [ ]:
# ============================================================
# ROC Curves (one-vs-rest, for both models)
# ============================================================
def plot_roc_curves(y_true, y_pred_prob, class_names, model_name, save_prefix):
    """Plot ROC curves for each class using one-vs-rest strategy."""
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=range(n_classes))

    plt.figure(figsize=(10, 8))
    colors = ['#E53935','#8E24AA','#1E88E5','#43A047','#FB8C00','#6D4C41']

    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=colors[i], lw=2,
                 label=f'{class_names[i]} (AUC = {roc_auc:.2f})')

    plt.plot([0,1],[0,1],'k--', lw=1.5, label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=11)
    plt.ylabel('True Positive Rate', fontsize=11)
    plt.title(f'{model_name} – ROC Curves (One-vs-Rest)', fontsize=13, fontweight='bold')
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{save_prefix}_roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_prefix}_roc_curves.png')

plot_roc_curves(cnn_true, cnn_probs, CLASS_NAMES, 'Custom CNN',  'custom_cnn')
plot_roc_curves(tl_true,  tl_probs,  CLASS_NAMES, 'MobileNetV2', 'mobilenetv2')

## Step 7: Bonus – Grad-CAM Visualisation
Grad-CAM highlights which regions of the image the CNN focuses on when making a prediction.

In [ ]:
# ============================================================
# Grad-CAM Implementation
# Shows which image regions influence the CNN decision
# ============================================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate Grad-CAM heatmap for a given image and model.
    img_array: preprocessed image with shape (1, H, W, C)
    """
    # Create gradient model
    grad_model = keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # Compute gradients
    grads = tape.gradient(class_channel, last_conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Weight conv output by pooled gradients
    last_conv_output = last_conv_output[0]
    heatmap = last_conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalise
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def display_gradcam(img_path, model, last_conv_layer_name, class_names, save_name):
    """Load image, generate Grad-CAM, and display overlay."""
    # Load and preprocess
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array_norm = np.expand_dims(img_array / 255.0, axis=0)

    # Prediction
    preds = model.predict(img_array_norm, verbose=0)
    pred_class = np.argmax(preds[0])
    pred_label = class_names[pred_class]
    confidence  = preds[0][pred_class] * 100

    # Grad-CAM
    heatmap = make_gradcam_heatmap(img_array_norm, model, last_conv_layer_name, pred_class)

    # Resize heatmap
    import cv2
    heatmap_resized = np.uint8(255 * heatmap)
    heatmap_resized = cv2.resize(heatmap_resized, (IMG_SIZE[1], IMG_SIZE[0]))
    heatmap_color = cv2.applyColorMap(heatmap_resized, cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)

    # Overlay
    overlay = np.uint8(img_array * 0.6 + heatmap_color * 0.4)

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(img_array.astype('uint8'))
    axes[0].set_title('Original Image', fontsize=10)
    axes[0].axis('off')

    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=10)
    axes[1].axis('off')

    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay\nPredicted: {pred_label} ({confidence:.1f}%)', fontsize=10)
    axes[2].axis('off')

    plt.suptitle(f'Grad-CAM Visualisation – Custom CNN', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_name, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_name}')


# Run Grad-CAM on sample images (one per class)
last_conv = 'conv3_2'   # Last conv layer in custom CNN

for class_name in CLASS_NAMES[:3]:   # Show 3 classes to save time
    cls_dir = os.path.join(DATASET_PATH, class_name)
    img_files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    if img_files:
        sample_path = os.path.join(cls_dir, img_files[5])  # Take 6th image
        display_gradcam(sample_path, cnn_model, last_conv, CLASS_NAMES,
                        f'gradcam_{class_name}.png')

## Step 8: Error Analysis – Misclassified Images

In [ ]:
# ============================================================
# Display misclassified images for the custom CNN
# ============================================================
val_generator.reset()
cnn_preds_flat = np.argmax(cnn_probs, axis=1)
misclassified_idx = np.where(cnn_preds_flat != cnn_true)[0]

print(f'Total misclassified: {len(misclassified_idx)} out of {len(cnn_true)}')

# Get all file paths
all_files = val_generator.filepaths

# Show first 8 misclassified images
show_n = min(8, len(misclassified_idx))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()

for i, idx in enumerate(misclassified_idx[:show_n]):
    img = plt.imread(all_files[idx])
    true_label = CLASS_NAMES[cnn_true[idx]]
    pred_label = CLASS_NAMES[cnn_preds_flat[idx]]
    axes[i].imshow(img)
    axes[i].set_title(f'True: {true_label}\nPred: {pred_label}',
                      color='red' if true_label != pred_label else 'green',
                      fontsize=9)
    axes[i].axis('off')

plt.suptitle('Misclassified Images – Custom CNN (Error Analysis)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: error_analysis.png')

## Step 9: Save Models & Summary

In [ ]:
# ============================================================
# Save final models
# ============================================================
cnn_model.save('custom_cnn_final.keras')
tl_model.save('mobilenetv2_final.keras')
print('Models saved: custom_cnn_final.keras, mobilenetv2_final.keras')

# ============================================================
# Final summary printout
# ============================================================
print('\n' + '='*55)
print('       FINAL RESULTS SUMMARY')
print('='*55)
print(f"  Custom CNN Accuracy    : {cnn_metrics[0]*100:.2f}%")
print(f"  Custom CNN F1-Score    : {cnn_metrics[3]*100:.2f}%")
print(f"  MobileNetV2 Accuracy   : {tl_metrics[0]*100:.2f}%")
print(f"  MobileNetV2 F1-Score   : {tl_metrics[3]*100:.2f}%")
print('='*55)
print('  Figures saved:')
figures = [
    'class_distribution.png', 'sample_images.png', 'augmentation_examples.png',
    'custom_cnn_training_curves.png', 'mobilenetv2_training_curves.png',
    'custom_cnn_confusion_matrix.png', 'mobilenetv2_confusion_matrix.png',
    'model_comparison.png', 'custom_cnn_roc_curves.png', 'mobilenetv2_roc_curves.png',
    'gradcam_*.png', 'error_analysis.png'
]
for f in figures:
    print(f'    - {f}')
print('='*55)